In [61]:
from pymongo import MongoClient
from pymongo.errors import DuplicateKeyError
from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError

In [62]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'sdk_information'

In [63]:
mongoDB_collection_2 = 'third_party_services'

In [64]:
def connect_mongodb(uri, db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def find_duplicate_maven_ids(collection):
    """
    Tìm tất cả các document có maven_id trùng nhau trong MongoDB.
    Trả về danh sách dict gồm:
    {
        'maven_id': <chuỗi>,
        'ids': [<id1>, <id2>, ...],
        'count': <số lượng trùng>
    }
    """
    pipeline = [
        {
            "$group": {
                "_id": "$maven_id",
                "ids": {"$push": "$_id"},
                "count": {"$sum": 1}
            }
        },
        {
            "$match": {
                "count": {"$gt": 1}  # chỉ lấy những maven_id có hơn 1 bản ghi
            }
        }
    ]

    results = []
    for doc in collection.aggregate(pipeline):
        results.append({
            "maven_id": doc["_id"],
            "ids": [str(_id) for _id in doc["ids"]],
            "count": doc["count"]
        })

    return results
def count_registration_types(collection):
    """
    Count how many documents have specific registration values:
    1. Empty string ("")
    2. "Owned by Google"
    3. "Registered in Google Play SDK Console"
    Returns a dictionary with counts.
    """
    counts = {
        "empty": collection.count_documents({"registration": ""}),
        "owned_by_google": collection.count_documents({"registration": "Owned by Google"}),
        "registered_in_console": collection.count_documents(
            {"registration": "Registered in Google Play SDK Console"}
        )
    }

    return counts


In [65]:
# col = connect_mongodb(mongoDB_uri, mongoDB_database, mongoDB_collection)

# duplicates = find_duplicate_maven_ids(col)

# for d in duplicates:
#     print(f"Maven ID: {d['maven_id']}")
#     print(f"Count: {d['count']}")
#     print(f"IDs: {d['ids']}")
#     print("------------------------------------------------")

In [66]:
col = connect_mongodb(mongoDB_uri, mongoDB_database, mongoDB_collection)

stats = count_registration_types(col)
print(stats)


{'empty': 88, 'owned_by_google': 18, 'registered_in_console': 171}


In [67]:
def get_unique_third_parties(collection):
    """
    Read all documents in the MongoDB collection and collect unique 'third_party' values.
    Returns a list of unique third_party names.
    """
    cursor = collection.find({}, {"third_party": 1, "_id": 0})  # only get 'third_party' field
    unique_parties = set()  # use set to automatically remove duplicates

    for doc in cursor:
        if "third_party" in doc and doc["third_party"]:  # check if field exists and not empty
            unique_parties.add(doc["third_party"].strip().lower())  # normalize casing

    return sorted(list(unique_parties))  # return sorted list for readability

col_2 = connect_mongodb(mongoDB_uri, mongoDB_database, mongoDB_collection_2)
third_party_list = get_unique_third_parties(col_2)
print(len(third_party_list))
print(third_party_list)

124
['acra', 'adapty', 'adjoe', 'adjust', 'adobe', 'adyen', 'agora', 'amap', 'amazon', 'appbrain', 'appier', 'applovin', 'appodeal', 'appsflyer', 'baidu', 'batch', 'bidmachine', 'bitmovin', 'blitzllama', 'blueshift', 'braintree', 'branch', 'braze', 'brightcove', 'bugsnag', 'bytedance openads', 'chartbeat', 'chartboost', 'clevertap', 'cometchat', 'contentsquare', 'customerio', 'cybersource', 'datadog', 'dexcom', 'digital turbine', 'expo', 'facebook', 'firebase', 'flurry', 'forter', 'freshchat', 'getstream', 'heap analytics', 'here', 'heytap push', 'huawei', 'hyprmx', 'iguang jpush', 'infobip', 'inmobi', 'instabug', 'intercom', 'ironsource', 'iterable', 'kakao', 'klaviyo', 'kochava', 'launchdarkly', 'leanplum', 'localytics', 'logrocket', 'lokalise', 'mapbox', 'matomo', 'mixpanel', 'mobilefuse', 'moengage', 'moovit', 'mparticle', 'naver', 'new relic', 'notifee', 'nuance', 'ogury', 'onesignal', 'optimizely', 'optimove', 'pangle bytedance', 'parse', 'picasso', 'plaid', 'pubnative', 'pushwoo

In [68]:
def get_maven_ids(collection):

    cursor = collection.find({}, {"maven_id": 1, "_id": 0})  # only get 'third_party' field
    unique_parties = set()  # use set to automatically remove duplicates

    for doc in cursor:
        if "maven_id" in doc and doc["maven_id"]:  # check if field exists and not empty
            unique_parties.add(doc["maven_id"].strip().lower())  # normalize casing

    return sorted(list(unique_parties))  # return sorted list for readability

maven_id_list = get_maven_ids(col)
print(len(maven_id_list))
print(maven_id_list)

277
['cn.thinkingdata.android:thinkinganalyticssdk', 'co.ogury:ogury-sdk', 'com.adcolony:sdk', 'com.adjust.sdk:adjust-android', 'com.admost.sdk:amr', 'com.adobe.marketing.mobile:core', 'com.adobe.marketing.mobile:sdk-core', 'com.adyen.checkout:checkout-core', 'com.adyen.checkout:drop-in', 'com.aliyun.dpa:oss-android-sdk', 'com.amazon.android:aps-admob-adapter', 'com.amazon.android:aps-sdk', 'com.amazon.device:amazon-appstore-sdk', 'com.amazonaws:aws-android-sdk-core', 'com.amazonaws:aws-android-sdk-sns', 'com.amazonaws:ivs-player', 'com.amplifyframework:aws-api', 'com.amplifyframework:core', 'com.amplitude:android-sdk', 'com.amplitude:experiment-android-client', 'com.android.billingclient:billing', 'com.anythink.sdk:core +7 more', 'com.anythink.sdk:core-tpn', 'com.apollographql.apollo3:apollo-runtime +6 more', 'com.apollographql.apollo:apollo-runtime +3 more', 'com.appboy:android-sdk-ui', 'com.appdynamics:appdynamics-runtime', 'com.applovin.mediation:adcolony-adapter', 'com.applovin.me

In [69]:
def find_unmatched_third_parties(third_party_list, maven_id_list):
    """
    Check if each third_party name is found (as a substring, case-insensitive)
    in any of the maven_id_list strings.
    
    Returns a list of third_party names that are NOT found in any maven_id.
    """
    unmatched = []

    # Normalize maven_id list for case-insensitive search
    maven_lower = [m.lower() for m in maven_id_list]

    for party in third_party_list:
        party_lower = party.lower()

        # Check if this third_party name is substring of any maven_id string
        found = any(party_lower in maven for maven in maven_lower)

        if not found:
            unmatched.append(party)

    return unmatched
unmatched = find_unmatched_third_parties(third_party_list, maven_id_list)
print("Unmatched third-party libraries:")
for u in unmatched:
    print("-", u)
print(f"\nTotal unmatched: {len(unmatched)}")

Unmatched third-party libraries:
- acra
- adapty
- adjoe
- amap
- appbrain
- appier
- baidu
- blitzllama
- blueshift
- bytedance openads
- chartbeat
- cometchat
- customerio
- cybersource
- dexcom
- digital turbine
- expo
- forter
- getstream
- heap analytics
- here
- heytap push
- iguang jpush
- infobip
- klaviyo
- logrocket
- mapbox
- matomo
- moovit
- new relic
- notifee
- nuance
- optimove
- pangle bytedance
- parse
- picasso
- pushwoosh
- pushy
- radar
- repro
- rollbar
- sailthru
- samsung health
- shakebugs
- sinch
- smart adserver
- smartlook
- survicate
- tasker
- transistorsoft
- unity ads
- urban airship
- validic
- vuforia
- zoho salesiq
- zoom

Total unmatched: 56


In [70]:
def get_third_party_by_sharing_status(collection):
    """
    Return 4 lists of 'third_party' names:
    1. stop_share_yes → documents with "stop_share": "yes"
    2. only_manifest_yes → documents with "manifest": "yes" and NOT "source_code": "yes"
    3. only_source_code_yes → documents with "source_code": "yes" and NOT "manifest": "yes"
    4. both_manifest_and_source_yes → documents with both "manifest" and "source_code" = "yes"
    """
    # 1️⃣ stop_share = yes
    stop_share_yes = [
        doc["third_party"] for doc in collection.find(
            {"stop_share": "yes"},
            {"third_party": 1, "_id": 0}
        )
        if "third_party" in doc
    ]

    # 2️⃣ manifest=yes, source_code != yes
    only_manifest_yes = [
        doc["third_party"] for doc in collection.find(
            {"manifest": "yes", "source_code": {"$ne": "yes"}},
            {"third_party": 1, "_id": 0}
        )
        if "third_party" in doc
    ]

    # 3️⃣ source_code=yes, manifest != yes
    only_source_code_yes = [
        doc["third_party"] for doc in collection.find(
            {"source_code": "yes", "manifest": {"$ne": "yes"}},
            {"third_party": 1, "_id": 0}
        )
        if "third_party" in doc
    ]

    # 4️⃣ both manifest=yes and source_code=yes
    both_manifest_and_source_yes = [
        doc["third_party"] for doc in collection.find(
            {"manifest": "yes", "source_code": "yes"},
            {"third_party": 1, "_id": 0}
        )
        if "third_party" in doc
    ]

    # Remove duplicates and return results
    return {
        "stop_share_yes": sorted(set(stop_share_yes)),
        "only_manifest_yes": sorted(set(only_manifest_yes)),
        "only_source_code_yes": sorted(set(only_source_code_yes)),
        "both_manifest_and_source_yes": sorted(set(both_manifest_and_source_yes))
    }

result = get_third_party_by_sharing_status(col_2)
# print(result)
# array_tps_has_stop_function = []
for key, values in result.items():
    print(f"\n--- {key} ({len(values)} items) ---")
    print(values)
#     array_tps_has_stop_function = array_tps_has_stop_function + values
# print(len(array_tps_has_stop_function))
# print(array_tps_has_stop_function)
array_tps_has_stop_function = ['ACRA', 'Adjust', 'Adobe', 'Adyen', 'Blueshift', 'Braintree', 'Chartboost', 
                               'Contentsquare', 'CustomerIO', 'Digital Turbine', 'Heap Analytics', 'HyprMX', 
                               'Instabug', 'Iterable', 'Kakao', 'Localytics', 'LogRocket', 'MParticle', 'Mixpanel', 
                               'MobileFuse', 'Naver', 'New Relic', 'PubNative', 'Sailthru', 'Samsung Health', 
                               'Singular', 'Smart AdServer', 'Smartlook', 'Taboola', 'UXCam', 'Unity Ads', 
                               'appLovin', 'appsflyer', 'baidu', 'batch', 'branch', 'braze', 'bugsnag', 'chartbeat', 
                               'cleverTap', 'facebook', 'firebase', 'ironsource', 'kochava', 'moengage', 'ogury', 
                               'onesignal', 'optimizely', 'pushwoosh', 'repro', 'sentry', 'smaato', 'swrve', 'urban airship', 'vungle']
unmatched_2 = find_unmatched_third_parties(array_tps_has_stop_function, maven_id_list)
print("Unmatched third-party libraries:")
for u in unmatched_2:
    print("-", u)
print(f"\nTotal unmatched 2: {len(unmatched_2)}")


--- stop_share_yes (55 items) ---
['ACRA', 'Adjust', 'Adobe', 'Adyen', 'Blueshift', 'Braintree', 'Chartboost', 'Contentsquare', 'CustomerIO', 'Digital Turbine', 'Heap Analytics', 'HyprMX', 'Instabug', 'Iterable', 'Kakao', 'Localytics', 'LogRocket', 'MParticle', 'Mixpanel', 'MobileFuse', 'Naver', 'New Relic', 'PubNative', 'Sailthru', 'Samsung Health', 'Singular', 'Smart AdServer', 'Smartlook', 'Taboola', 'UXCam', 'Unity Ads', 'appLovin', 'appsflyer', 'baidu', 'batch', 'branch', 'braze', 'bugsnag', 'chartbeat', 'cleverTap', 'facebook', 'firebase', 'ironsource', 'kochava', 'moengage', 'ogury', 'onesignal', 'optimizely', 'pushwoosh', 'repro', 'sentry', 'smaato', 'swrve', 'urban airship', 'vungle']

--- only_manifest_yes (1 items) ---
['repro']

--- only_source_code_yes (44 items) ---
['Adjust', 'Adobe', 'Adyen', 'Blueshift', 'Braintree', 'Chartboost', 'Contentsquare', 'CustomerIO', 'Digital Turbine', 'Heap Analytics', 'HyprMX', 'Instabug', 'Iterable', 'Kakao', 'Localytics', 'LogRocket', '